# Heart Rate Estimation from NIR Facial Videos

This notebook builds a simple neural network that predicts heart rate (in BPM) from Near-Infrared facial video clips.

Dataset used: **MR-NIRP-D**

## Plan:
1. Load and explore the dataset
2. Extract face ROI from NIR frames
3. Prepare ground truth BPM from reference signals
4. Build a 3D CNN model
5. Train the model
6. Evaluate with MAE, RMSE, Pearson correlation

## Step 1 — Install & Import Libraries

In [ ]:
# install dependencies if needed
# !pip install torch torchvision opencv-python scipy matplotlib

In [ ]:
import os
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.signal import butter, sosfiltfilt, find_peaks, welch
from scipy.stats import pearsonr
from scipy.io import loadmat

print('All imports done!')
print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Step 2 — Set the Dataset Path

Point this to wherever you downloaded MR-NIRP-D.

Expected folder structure:
```
MR-NIRP-D/
  subject_01/
    session_01/
      nir/         <-- NIR frames (*.png) or video
      reference/   <-- .mat or .csv reference signal
```

In [ ]:
# UPDATE THIS to your dataset path
DATASET_ROOT = './MR-NIRP-D'

# Clip settings
CLIP_LEN  = 150   # 5 seconds at 30fps
STRIDE    = 75    # 50% overlap between clips
ROI_SIZE  = 64    # resize face to 64x64
BATCH_SIZE = 8
EPOCHS     = 30
LR         = 1e-3

print('Settings:')
print(f'  Dataset root : {DATASET_ROOT}')
print(f'  Clip length  : {CLIP_LEN} frames')
print(f'  ROI size     : {ROI_SIZE}x{ROI_SIZE}')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  Epochs       : {EPOCHS}')

## Step 3 — Ground Truth: Deriving BPM from Reference Signal

The MR-NIRP-D dataset provides a contact sensor (PPG) signal alongside the video.
We need to convert this raw signal into a BPM value for each clip.

**Approach:**
1. Bandpass filter between 0.7–3.5 Hz (= 42–210 BPM range)
2. Find peaks in the filtered signal
3. Compute mean time between peaks → BPM
4. Fallback to Welch PSD if peaks are unclear

In [ ]:
def bandpass_filter(signal, fs, lowcut=0.7, highcut=3.5, order=2):
    """Bandpass filter using 2nd-order SOS (numerically stable at low Hz)."""
    nyq = 0.5 * fs
    sos = butter(order, [lowcut / nyq, highcut / nyq], btype='band', output='sos')
    return sosfiltfilt(sos, signal)


def derive_bpm(signal, fs):
    """
    Compute BPM from raw reference signal.
    Uses peak detection, falls back to Welch PSD.
    """
    # filter to HR range
    filtered = bandpass_filter(signal, fs)
    filtered = (filtered - filtered.mean()) / (filtered.std() + 1e-8)

    # try peak detection first
    min_dist = int(fs * 0.3)  # 200 BPM max
    peaks, _ = find_peaks(filtered, distance=min_dist)

    if len(peaks) >= 2:
        ibi = np.diff(peaks) / fs   # inter-beat intervals in seconds
        bpm = 60.0 / ibi.mean()
        return float(np.clip(bpm, 40, 210))

    # fallback: Welch power spectral density
    nperseg = min(int(fs * 4), len(filtered))
    freqs, psd = welch(filtered, fs=fs, nperseg=nperseg)
    mask = (freqs >= 0.7) & (freqs <= 3.5)
    if mask.sum() == 0:
        return 70.0  # safe default if something goes wrong
    peak_freq = freqs[mask][np.argmax(psd[mask])]
    return float(np.clip(peak_freq * 60.0, 40, 210))


print('Functions defined!')

In [ ]:
# Quick test — simulate a 72 BPM signal and see if we recover it
fs_test = 1000.0
t_test  = np.arange(0, 15, 1 / fs_test)
fake_ppg = np.sin(2 * np.pi * 1.2 * t_test) + 0.1 * np.random.randn(len(t_test))

recovered_bpm = derive_bpm(fake_ppg, fs=fs_test)
print(f'Simulated signal: 72 BPM')
print(f'Recovered BPM:    {recovered_bpm:.1f} BPM')

# plot the filtered signal
filtered_test = bandpass_filter(fake_ppg, fs=fs_test)
plt.figure(figsize=(12, 3))
plt.plot(t_test[:3000], filtered_test[:3000])
plt.title('Filtered PPG signal (first 3 seconds)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

## Step 4 — Face ROI Extraction from NIR Frames

We use OpenCV's Haar cascade to detect the face and crop it to a fixed size.
If face detection fails on a frame, we reuse the last known bounding box.

In [ ]:
class FaceROIExtractor:
    """Detects face in NIR frames and returns a normalized square crop."""

    def __init__(self, roi_size=64):
        self.roi_size = roi_size
        cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        self.detector = cv2.CascadeClassifier(cascade_path)
        self._last_bbox = None  # remember last good detection

    def extract(self, frame):
        """
        Input:  grayscale NIR frame (H, W)
        Output: float32 array (roi_size, roi_size) in [0, 1]
        """
        gray = frame if frame.ndim == 2 else frame[:, :, 0]

        faces = self.detector.detectMultiScale(
            gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60)
        )

        if len(faces) > 0:
            # take the largest detected face
            areas = [w * h for (x, y, w, h) in faces]
            x, y, w, h = faces[np.argmax(areas)]
            self._last_bbox = (x, y, w, h)
        elif self._last_bbox is not None:
            x, y, w, h = self._last_bbox  # reuse previous frame's bbox
        else:
            # no face ever found — use centre crop
            H, W = gray.shape
            m = min(H, W) // 4
            x, y, w, h = m, m, W - 2*m, H - 2*m

        roi = gray[y:y+h, x:x+w]
        roi = cv2.resize(roi, (self.roi_size, self.roi_size))
        return roi.astype(np.float32) / 255.0


print('FaceROIExtractor ready!')

## Step 5 — Dataset Class

This loads the NIR video frames, extracts face ROIs, and pairs each clip with a BPM label from the reference signal.

In [ ]:
def load_reference_signal(ref_path):
    """Load .mat or .csv reference physiological signal. Returns (signal, fs)."""
    ext = os.path.splitext(ref_path)[-1].lower()
    if ext == '.mat':
        mat = loadmat(ref_path)
        for key in ['ppg', 'bvp', 'pulse', 'hr_signal']:
            if key in mat:
                fs = float(mat.get('fs', np.array([[1000.0]]))[0][0])
                return mat[key].flatten().astype(np.float64), fs
        # fallback — grab first numeric array
        for k, v in mat.items():
            if not k.startswith('_') and isinstance(v, np.ndarray) and v.ndim <= 2:
                return v.flatten().astype(np.float64), 1000.0
    elif ext == '.csv':
        data = np.loadtxt(ref_path, delimiter=',', skiprows=1)
        signal = data[:, -1] if data.ndim > 1 else data
        return signal.astype(np.float64), 1000.0
    raise ValueError(f'Unsupported format: {ext}')


class NIRHeartRateDataset(Dataset):
    """
    Loads sliding-window clips from MR-NIRP-D NIR sessions.
    Each sample: (clip_tensor, bpm_label)
    clip_tensor shape: (1, T, H, W)  — single channel, T frames
    """

    def __init__(self, root_dir, clip_len=150, stride=75, roi_size=64,
                 split='train', split_ratio=(0.7, 0.15)):
        self.clip_len = clip_len
        self.stride   = stride
        self.roi_size = roi_size
        self.extractor = FaceROIExtractor(roi_size)

        # discover all clips
        all_clips = self._discover(root_dir)

        # split by clip index (deterministic)
        n = len(all_clips)
        n_train = int(n * split_ratio[0])
        n_val   = int(n * split_ratio[1])
        if split == 'train':
            self.clips = all_clips[:n_train]
        elif split == 'val':
            self.clips = all_clips[n_train:n_train + n_val]
        else:
            self.clips = all_clips[n_train + n_val:]

    def _discover(self, root_dir):
        """Walk directory tree, build list of (frame_list, ref_path, start_idx)."""
        clips = []
        for subject in sorted(os.listdir(root_dir)):
            subj_path = os.path.join(root_dir, subject)
            if not os.path.isdir(subj_path):
                continue
            for session in sorted(os.listdir(subj_path)):
                sess_path = os.path.join(subj_path, session)
                nir_path  = os.path.join(sess_path, 'nir')
                ref_dir   = os.path.join(sess_path, 'reference')
                if not os.path.exists(nir_path):
                    continue

                # get frame list
                frames = []
                for ext in ('*.png', '*.jpg', '*.bmp'):
                    frames += glob.glob(os.path.join(nir_path, ext))
                frames = sorted(frames)
                if len(frames) < self.clip_len:
                    continue

                # find reference signal
                ref_files = (glob.glob(os.path.join(ref_dir, '*.mat')) +
                             glob.glob(os.path.join(ref_dir, '*.csv')))
                ref_path = ref_files[0] if ref_files else None

                # sliding window clips
                for start in range(0, len(frames) - self.clip_len + 1, self.stride):
                    clips.append((frames, ref_path, start))
        return clips

    def _load_clip(self, frames, start):
        rois = []
        for fp in frames[start:start + self.clip_len]:
            img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
            if img is None:
                img = np.zeros((self.roi_size, self.roi_size), dtype=np.uint8)
            rois.append(self.extractor.extract(img))
        return np.stack(rois, axis=0)  # (T, H, W)

    def _load_bpm(self, ref_path, start, total_frames):
        if ref_path is None:
            return 70.0
        signal, fs = load_reference_signal(ref_path)
        # align reference window to clip position
        ref_start = int(start / total_frames * len(signal))
        ref_end   = int((start + self.clip_len) / total_frames * len(signal))
        clip_sig  = signal[ref_start:min(ref_end, len(signal))]
        if len(clip_sig) < 10:
            return 70.0
        return derive_bpm(clip_sig, fs)

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        frames, ref_path, start = self.clips[idx]
        clip = self._load_clip(frames, start)                  # (T, H, W)
        clip_tensor = torch.from_numpy(clip).unsqueeze(0)      # (1, T, H, W)
        bpm = self._load_bpm(ref_path, start, len(frames))
        return clip_tensor, torch.tensor(bpm, dtype=torch.float32)


print('Dataset class defined!')

In [ ]:
# Load datasets
print('Loading train/val/test datasets...')

train_ds = NIRHeartRateDataset(DATASET_ROOT, CLIP_LEN, STRIDE, ROI_SIZE, split='train')
val_ds   = NIRHeartRateDataset(DATASET_ROOT, CLIP_LEN, STRIDE, ROI_SIZE, split='val')
test_ds  = NIRHeartRateDataset(DATASET_ROOT, CLIP_LEN, STRIDE, ROI_SIZE, split='test')

print(f'Train clips : {len(train_ds)}')
print(f'Val clips   : {len(val_ds)}')
print(f'Test clips  : {len(test_ds)}')

# peek at one sample
if len(train_ds) > 0:
    sample_clip, sample_bpm = train_ds[0]
    print(f'\nSample clip shape : {sample_clip.shape}')   # (1, T, H, W)
    print(f'Sample BPM label  : {sample_bpm.item():.1f}')

In [ ]:
# Visualise a few frames from one clip
if len(train_ds) > 0:
    clip, bpm = train_ds[0]
    frames_to_show = [0, 30, 60, 90, 120]

    fig, axes = plt.subplots(1, len(frames_to_show), figsize=(15, 3))
    for i, t in enumerate(frames_to_show):
        axes[i].imshow(clip[0, t].numpy(), cmap='gray')
        axes[i].set_title(f'Frame {t}')
        axes[i].axis('off')
    plt.suptitle(f'NIR Face ROI — Ground Truth BPM: {bpm.item():.1f}')
    plt.tight_layout()
    plt.show()

In [ ]:
# look at BPM distribution in training set
if len(train_ds) > 0:
    bpms = [train_ds[i][1].item() for i in range(min(len(train_ds), 200))]
    plt.figure(figsize=(8, 4))
    plt.hist(bpms, bins=20, color='steelblue', edgecolor='white')
    plt.xlabel('Heart Rate (BPM)')
    plt.ylabel('Count')
    plt.title('BPM Distribution in Training Set')
    plt.tight_layout()
    plt.show()
    print(f'Mean BPM : {np.mean(bpms):.1f}')
    print(f'Std BPM  : {np.std(bpms):.1f}')
    print(f'Min/Max  : {np.min(bpms):.1f} / {np.max(bpms):.1f}')

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')

## Step 6 — Model Architecture

We use a **3D CNN + Transformer** model:
- 3D CNN extracts local spatial+temporal features (captures subtle blood flow motion)
- Transformer encoder captures long-range periodicity across frames
- Final head predicts a single BPM value

Inspired by PhysFormer (Yu et al., CVPR 2022) but much simpler — ~1.2M params.

In [ ]:
import math

# --- CNN Stem ---
class ConvBnRelu3D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=(1,3,3), stride=(1,1,1), padding=(0,1,1)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel, stride=stride, padding=padding, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)


class CNN3DStem(nn.Module):
    """
    3D CNN that reduces spatial dims while keeping temporal resolution.
    Input:  (B, 1, T, 64, 64)
    Output: (B, T, 128)
    """
    def __init__(self, base_ch=32):
        super().__init__()
        self.net = nn.Sequential(
            ConvBnRelu3D(1,          base_ch,   kernel=(1,5,5), padding=(0,2,2)),
            nn.MaxPool3d((1,2,2)),
            ConvBnRelu3D(base_ch,    base_ch*2),
            nn.MaxPool3d((1,2,2)),
            ConvBnRelu3D(base_ch*2,  base_ch*4),
            nn.MaxPool3d((1,2,2)),
        )
        self.pool = nn.AdaptiveAvgPool3d((None, 1, 1))   # keep T, collapse H,W
        self.out_ch = base_ch * 4

    def forward(self, x):
        x = self.net(x)             # (B, C, T, h, w)
        x = self.pool(x)            # (B, C, T, 1, 1)
        x = x.squeeze(-1).squeeze(-1)   # (B, C, T)
        return x.permute(0, 2, 1)   # (B, T, C)


# --- Positional Encoding ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


# --- Full Model ---
class HREstimator(nn.Module):
    """
    NIR Video → BPM
    3D CNN Stem → Transformer → BPM regression head
    """
    def __init__(self, base_ch=32, n_heads=4, n_layers=2, bpm_min=40.0, bpm_max=210.0):
        super().__init__()
        d_model = base_ch * 4

        self.cnn  = CNN3DStem(base_ch)
        self.pe   = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=256,
            dropout=0.1, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
        self.bpm_min = bpm_min
        self.bpm_max = bpm_max

    def forward(self, x):
        x = self.cnn(x)                     # (B, T, d_model)
        x = self.pe(x)
        x = self.transformer(x)             # (B, T, d_model)
        x = x.mean(dim=1)                   # (B, d_model) — average over time
        x = self.head(x).squeeze(-1)        # (B,)
        # clamp output to valid BPM range using sigmoid
        return torch.sigmoid(x) * (self.bpm_max - self.bpm_min) + self.bpm_min


# create model
model = HREstimator(base_ch=32, n_heads=4, n_layers=2).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model created!')
print(f'Total trainable parameters: {total_params:,}')

# quick forward pass sanity check
dummy = torch.randn(2, 1, CLIP_LEN, ROI_SIZE, ROI_SIZE).to(device)
with torch.no_grad():
    out = model(dummy)
print(f'Input shape  : {dummy.shape}')
print(f'Output shape : {out.shape}  →  values: {out.cpu().numpy().round(1)}')

## Step 7 — Loss Function & Optimizer

- **Loss:** Huber loss (also called SmoothL1). Better than MSE because it's less sensitive to outlier BPM values caused by noisy reference signals.
- **Optimizer:** AdamW with cosine LR decay

In [ ]:
criterion = nn.HuberLoss(delta=5.0)   # delta=5 BPM
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

print('Loss     : HuberLoss (delta=5 BPM)')
print('Optimizer: AdamW (lr=1e-3, weight_decay=1e-4)')
print('Scheduler: CosineAnnealingLR')

## Step 8 — Training

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    all_losses, all_preds, all_targets = [], [], []

    for clips, bpms in loader:
        clips = clips.to(device)
        bpms  = bpms.to(device)

        optimizer.zero_grad()
        preds = model(clips)
        loss  = criterion(preds, bpms)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()

        all_losses.append(loss.item())
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(bpms.cpu().numpy())

    mae = np.mean(np.abs(np.array(all_preds) - np.array(all_targets)))
    return np.mean(all_losses), mae


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    all_losses, all_preds, all_targets = [], [], []

    for clips, bpms in loader:
        clips = clips.to(device)
        bpms  = bpms.to(device)
        preds = model(clips)
        loss  = criterion(preds, bpms)

        all_losses.append(loss.item())
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(bpms.cpu().numpy())

    mae = np.mean(np.abs(np.array(all_preds) - np.array(all_targets)))
    return np.mean(all_losses), mae


print('Training functions defined!')

In [ ]:
# Training loop!
train_losses, val_losses = [], []
train_maes,   val_maes   = [], []

best_val_mae = float('inf')
best_epoch   = 0
patience     = 10
patience_ctr = 0

os.makedirs('outputs', exist_ok=True)

print(f'Starting training for {EPOCHS} epochs...\n')
print(f'{"Epoch":>6}  {"Train Loss":>10}  {"Val Loss":>8}  {"Train MAE":>10}  {"Val MAE":>8}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_mae = train_one_epoch(model, train_loader, optimizer, criterion, device)
    vl_loss, vl_mae = validate(model, val_loader, criterion, device)
    scheduler.step()

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    train_maes.append(tr_mae)
    val_maes.append(vl_mae)

    print(f'{epoch:>6}  {tr_loss:>10.4f}  {vl_loss:>8.4f}  {tr_mae:>10.2f}  {vl_mae:>8.2f}')

    # save best model
    if vl_mae < best_val_mae:
        best_val_mae = vl_mae
        best_epoch   = epoch
        patience_ctr = 0
        torch.save(model.state_dict(), 'outputs/best_model.pth')
        print(f'         --> saved best model (val MAE = {best_val_mae:.2f} BPM)')
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f'\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)')
            break

print(f'\nTraining done! Best val MAE: {best_val_mae:.2f} BPM at epoch {best_epoch}')

## Step 9 — Plot Training Curves

In [ ]:
epochs_ran = range(1, len(train_losses) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_ran, train_losses, label='Train Loss', color='steelblue')
ax1.plot(epochs_ran, val_losses,   label='Val Loss',   color='tomato')
ax1.set_title('Huber Loss over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_ran, train_maes, label='Train MAE', color='steelblue')
ax2.plot(epochs_ran, val_maes,   label='Val MAE',   color='tomato')
ax2.axvline(best_epoch, linestyle='--', color='gray', alpha=0.7, label=f'Best epoch ({best_epoch})')
ax2.set_title('MAE (BPM) over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE (BPM)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150)
plt.show()
print('Saved training curves to outputs/training_curves.png')

## Step 10 — Evaluation on Test Set

Metrics used:
- **MAE** — Mean Absolute Error in BPM (most intuitive)
- **RMSE** — Root Mean Square Error (penalises large errors more)
- **Pearson r** — Correlation between predicted and ground truth BPM

In [ ]:
# load best checkpoint
model.load_state_dict(torch.load('outputs/best_model.pth', map_location=device))
model.eval()
print('Loaded best model checkpoint')

# run on test set
all_preds, all_targets = [], []

with torch.no_grad():
    for clips, bpms in test_loader:
        clips = clips.to(device)
        preds = model(clips).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(bpms.numpy())

all_preds   = np.array(all_preds)
all_targets = np.array(all_targets)

# compute metrics
mae     = np.mean(np.abs(all_preds - all_targets))
rmse    = np.sqrt(np.mean((all_preds - all_targets) ** 2))
pearson = pearsonr(all_preds, all_targets)[0] if len(all_preds) > 1 else float('nan')

print('\n==============================')
print('     TEST SET RESULTS')
print('==============================')
print(f'  MAE     : {mae:.2f} BPM')
print(f'  RMSE    : {rmse:.2f} BPM')
print(f'  Pearson : {pearson:.4f}')
print('==============================')

In [ ]:
# Scatter plot: predicted vs ground truth
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(all_targets, all_preds, alpha=0.6, edgecolors='k', linewidths=0.4, s=40, color='steelblue')

lim_min = min(all_targets.min(), all_preds.min()) - 5
lim_max = max(all_targets.max(), all_preds.max()) + 5
ax.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', linewidth=1.5, label='Perfect prediction')

ax.set_xlabel('Ground Truth BPM')
ax.set_ylabel('Predicted BPM')
ax.set_title(f'Predicted vs Ground Truth HR\nMAE={mae:.2f} BPM | RMSE={rmse:.2f} | r={pearson:.3f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/pred_vs_gt.png', dpi=150)
plt.show()

In [ ]:
# Error distribution
errors = all_preds - all_targets

plt.figure(figsize=(8, 4))
plt.hist(errors, bins=30, color='steelblue', edgecolor='white')
plt.axvline(0, color='red', linestyle='--', label='Zero error')
plt.axvline(mae,  color='orange', linestyle='--', label=f'Mean AE = {mae:.1f} BPM')
plt.axvline(-mae, color='orange', linestyle='--')
plt.xlabel('Prediction Error (BPM)')
plt.ylabel('Count')
plt.title('Error Distribution')
plt.legend()
plt.tight_layout()
plt.show()

## Step 11 — Summary & Next Steps

### What we built:
- Loaded NIR facial video from MR-NIRP-D
- Detected face with Haar cascade → cropped 64×64 ROI per frame
- Derived ground truth BPM from reference PPG using bandpass filtering + peak detection
- Built a 3D CNN + Transformer model to regress BPM from 5-second video clips
- Evaluated with MAE, RMSE, and Pearson correlation

### Limitations:
- Haar cascade can fail on NIR if the model wasn't trained on NIR images — MediaPipe or a NIR-specific detector would work better
- Small dataset → model may overfit; cross-subject validation would be more rigorous
- Only doing BPM regression; predicting the full rPPG waveform first and then extracting BPM could improve accuracy

### What I'd try next:
- Add data augmentation (temporal jitter, Gaussian noise, flip)
- Try leave-one-subject-out cross validation
- Add a frequency-domain loss on the temporal feature signal
- Use a pre-trained 3D ResNet and fine-tune it

In [ ]:
# Final summary
print('=' * 45)
print('FINAL RESULTS SUMMARY')
print('=' * 45)
print(f'Model          : 3D CNN + Transformer')
print(f'Parameters     : {total_params:,}')
print(f'Clip length    : {CLIP_LEN} frames (5s @ 30fps)')
print(f'Best epoch     : {best_epoch}')
print(f'Best val MAE   : {best_val_mae:.2f} BPM')
print(f'Test MAE       : {mae:.2f} BPM')
print(f'Test RMSE      : {rmse:.2f} BPM')
print(f'Test Pearson r : {pearson:.4f}')
print('=' * 45)